In [ ]:
# ЯЧЕЙКА 1: УСТАНОВКА БИБЛИОТЕК
!pip install python-docx pdfminer.six requests tqdm spacy mistralai python-dotenv google-colab
!python -m spacy download ru_core_news_sm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 42.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.8/442.8 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.3/160.3 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 39.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.3/15.3 MB 106.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 72.0 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('ru_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
# ЯЧЕЙКА 2: ИМПОРТ БИБЛИОТЕК
import os
import json
import logging
import time
import csv
import re
from datetime import datetime, timedelta
from pathlib import Path
from typing import List, Dict, Any, Optional
from urllib.parse import urlparse
import signal
from contextlib import contextmanager

# Импортируем библиотеки для работы с документами
from docx import Document as DocxDocument
from pdfminer.high_level import extract_text as pdf_extract_text

# Импортируем библиотеки для работы с API
import requests

# Импортируем spaCy для NLP
import spacy
from spacy.tokens import Doc

# Импортируем библиотеку для работы с Mistral
from mistralai import Mistral

# Настройка логирования
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler("tender_processing.log"),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger("tender_processing")

In [ ]:
# ЯЧЕЙКА 3: КЛАСС ДЛЯ РАБОТЫ С API ZAKUPKI360 (ПЕРЕРАБОТАННЫЙ)
class Zakupki360APIClient:
    """Корректный клиент для работы с API Zakupki360 на основе Swagger"""

    # Базовые URL для разных сред
    BASE_URLS = {
        "prod": "https://api.zakupki360.ru",
        "test": "https://api.zakupki360.ru"  # Уточнить тестовый URL
    }

    def __init__(self, login: str, password: str, environment: str = "prod"):
        self.login = login
        self.password = password
        self.environment = environment
        self.base_url = self.BASE_URLS.get(environment, self.BASE_URLS["prod"])
        self.access_token = None
        self.token_expires = None
        self.requests_count = {"search": 0, "orders": 0, "customers": 0, "suppliers": 0, "documents": 0}
        self.daily_limits = {"search": 100, "orders": 500, "customers": 500, "suppliers": 500, "documents": 5000}
        self.logger = logging.getLogger(f"{__name__}.Zakupki360APIClient")
        self.session = requests.Session()
        self._authenticate()

    def _authenticate(self):
        """Аутентификация по API согласно Swagger"""
        auth_url = f"{self.base_url}/token"
        payload = {
            "login": self.login,
            "password": self.password
        }
        headers = {
            "Content-Type": "application/json-patch+json"
        }

        self.logger.info("🔐 Аутентификация в Zakupki360 API...")
        try:
            response = self.session.post(auth_url, json=payload, headers=headers, timeout=30)

            if response.status_code == 200:
                auth_data = response.json()
                self.access_token = auth_data.get("access_token") or auth_data.get("token")

                if not self.access_token:
                    self.logger.error(f"❌ Токен не найден в ответе. Полный ответ: {auth_data}")
                    raise ValueError("Токен не получен в ответе от API")

                expires_in = auth_data.get("expires_in", 3600)
                self.token_expires = datetime.now() + timedelta(seconds=expires_in)
                self.logger.info("✅ Аутентификация успешна!")
                self.logger.info(f"Токен получен, действителен до: {self.token_expires}")

                # Сохраняем токен в заголовки сессии
                self.session.headers.update({
                    "Authorization": f"Bearer {self.access_token}",
                    "Content-Type": "application/json",
                    "Accept": "application/json"
                })
            else:
                self.logger.error(f"❌ Ошибка аутентификации: {response.status_code}")
                self.logger.error(f"Ответ сервера: {response.text}")
                response.raise_for_status()

        except requests.exceptions.RequestException as e:
            self.logger.error(f"❌ Сетевая ошибка при аутентификации: {str(e)}")
            raise
        except Exception as e:
            self.logger.error(f"❌ Ошибка аутентификации: {str(e)}")
            raise

    def _check_rate_limit(self, endpoint_type: str) -> bool:
        """Проверка лимитов запросов"""
        if self.requests_count[endpoint_type] >= self.daily_limits[endpoint_type]:
            self.logger.warning(f"⚠️ Достигнут дневной лимит для {endpoint_type}: {self.daily_limits[endpoint_type]}")
            return False
        return True

    def _make_request(self, method: str, endpoint: str, endpoint_type: str, params=None, data=None):
        """Универсальный метод для выполнения запросов с учетом лимитов"""
        if not self._check_rate_limit(endpoint_type):
            raise Exception(f"Достигнут дневной лимит запросов для {endpoint_type}")

        # Обновляем токен если истек
        if not self.access_token or (self.token_expires and datetime.now() >= self.token_expires):
            self._authenticate()

        url = f"{self.base_url}{endpoint}"

        self.logger.debug(f"Выполнение запроса: {method} {url}")
        self.logger.debug(f"Параметры: {params}")

        try:
            if method.upper() == "GET":
                response = self.session.get(url, params=params, timeout=30)
            elif method.upper() == "POST":
                response = self.session.post(url, json=data, params=params, timeout=30)
            else:
                raise ValueError(f"Неподдерживаемый HTTP метод: {method}")

            self.logger.debug(f"Статус ответа: {response.status_code}")

            if response.status_code == 200:
                self.requests_count[endpoint_type] += 1
                self.logger.debug(f"✅ Запрос {endpoint_type} выполнен успешно")
                return response.json()
            elif response.status_code == 401:
                self.logger.warning("⚠️ Токен истек, пытаемся обновить...")
                self._authenticate()
                return self._make_request(method, endpoint, endpoint_type, params, data)
            else:
                self.logger.error(f"❌ Ошибка запроса {endpoint_type}: {response.status_code}")
                self.logger.error(f"URL: {url}")
                self.logger.error(f"Ответ сервера: {response.text}")
                response.raise_for_status()

        except requests.exceptions.RequestException as e:
            self.logger.error(f"❌ Сетевая ошибка при запросе {endpoint_type}: {str(e)}")
            raise

    def search_tenders(self, search_params: Dict[str, Any]) -> List[Dict]:
        """Поиск тендеров с корректными параметрами согласно Swagger"""
        self.logger.info(f"🔍 Поиск тендеров с параметрами: {search_params}")

        # Базовые параметры для поиска
        params = {
            "page": search_params.get("page", 1),
            "per_page": min(search_params.get("per_page", 20), 100),  # Ограничиваем для тестов
        }

        # Добавляем поисковый запрос если есть
        if search_params.get("query"):
            params["query"] = search_params["query"]

        # Добавляем фильтры по дате если указаны
        if search_params.get("date_from"):
            params["dateFrom"] = search_params["date_from"]
        if search_params.get("date_to"):
            params["dateTo"] = search_params["date_to"]

        try:
            result = self._make_request("GET", "/api/orders/search", "search", params=params)

            # Адаптивная обработка ответа
            if isinstance(result, dict):
                tenders = result.get("data") or result.get("orders") or result.get("items") or []
            elif isinstance(result, list):
                tenders = result
            else:
                tenders = []

            self.logger.info(f"✅ Найдено {len(tenders)} тендеров")

            # Логируем структуру для отладки
            if tenders and len(tenders) > 0:
                self.logger.debug(f"📋 Структура первого тендера: {list(tenders[0].keys())}")

            return tenders

        except Exception as e:
            self.logger.error(f"❌ Ошибка при поиске тендеров: {str(e)}")
            return []

    def search_tenders_advanced(self, filters: Dict[str, Any]) -> List[Dict]:
        """Расширенный поиск с фильтрами"""
        self.logger.info(f"🔍 Расширенный поиск с фильтрами: {filters}")
        return self.search_tenders(filters)

    def get_tender_details(self, tender_id: str) -> Dict:
        """Получение детальной информации о тендере"""
        self.logger.info(f"📋 Получение деталей тендера {tender_id}...")

        try:
            result = self._make_request("GET", f"/api/orders/{tender_id}", "orders")
            return result
        except Exception as e:
            self.logger.error(f"❌ Ошибка при получении деталей тендера {tender_id}: {str(e)}")
            return {}

    def test_connection(self) -> bool:
        """Тестирование подключения к API"""
        try:
            tenders = self.search_tenders({"per_page": 1})
            return len(tenders) >= 0  # Если нет исключения - подключение работает
        except Exception as e:
            self.logger.error(f"❌ Тест подключения не пройден: {str(e)}")
            return False

    def get_usage_stats(self) -> Dict:
        """Получение статистики использования API"""
        return {
            "limits": self.daily_limits,
            "used": self.requests_count,
            "remaining": {
                endpoint: self.daily_limits[endpoint] - self.requests_count[endpoint]
                for endpoint in self.daily_limits
            }
        }

In [ ]:
# ЯЧЕЙКА 4: КЛАСС ДЛЯ ЗАГРУЗКИ ФАЙЛОВ
class FileDownloader:
    """Класс для загрузки файлов по внешним ссылкам"""

    def __init__(self):
        self.logger = logging.getLogger(f"{__name__}.FileDownloader")
        self.session = requests.Session()
        self.session.headers.update({
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
        })

    def download_file(self, file_url: str, timeout: int = 30) -> str:
        """Загрузка файла по внешней ссылке с обработкой ошибок"""
        try:
            self.logger.info(f"📥 Загрузка файла: {file_url}")

            # Обработка Google Docs ссылок
            if "docs.google.com" in file_url and "/document/" in file_url:
                return self._download_google_docs(file_url, timeout)
            else:
                return self._download_regular_file(file_url, timeout)

        except Exception as e:
            self.logger.error(f"❌ Ошибка при загрузке файла: {str(e)}")
            raise

    def _download_google_docs(self, file_url: str, timeout: int) -> str:
        """Скачивание Google Docs документа"""
        # Извлекаем ID документа
        match = re.search(r'/d/([a-zA-Z0-9-_]+)', file_url)
        if not match:
            raise ValueError("Неверный формат ссылки на Google Docs")

        doc_id = match.group(1)
        export_url = f"https://docs.google.com/document/d/{doc_id}/export?format=txt"

        self.logger.info(f"📄 Скачивание Google Docs документа: {doc_id}")
        response = self.session.get(export_url, timeout=timeout)

        if response.status_code == 200:
            filename = f"company_description_{doc_id}_{int(time.time())}.txt"
            with open(filename, 'wb') as f:
                f.write(response.content)
            self.logger.info(f"✅ Google Docs документ сохранен как: {filename}")
            return filename
        else:
            raise Exception(f"❌ Ошибка загрузки Google Docs: {response.status_code}")

    def _download_regular_file(self, file_url: str, timeout: int) -> str:
        """Скачивание обычного файла"""
        parsed_url = urlparse(file_url)
        original_filename = os.path.basename(parsed_url.path)

        if not original_filename:
            original_filename = f"downloaded_file_{int(time.time())}"

        # Очищаем имя файла от недопустимых символов
        clean_filename = re.sub(r'[<>:\"/\\|?*]', '_', original_filename)

        self.logger.info(f"📄 Скачивание файла: {clean_filename}")
        response = self.session.get(file_url, timeout=timeout, stream=True)

        if response.status_code == 200:
            with open(clean_filename, 'wb') as f:
                for chunk in response.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)
            self.logger.info(f"✅ Файл сохранен как: {clean_filename}")
            return clean_filename
        else:
            raise Exception(f"❌ Ошибка загрузки файла: {response.status_code}")

    def validate_file(self, file_path: str) -> bool:
        """Проверка валидности загруженного файла"""
        if not os.path.exists(file_path):
            return False

        file_size = os.path.getsize(file_path)
        if file_size == 0:
            return False

        return True

In [ ]:
# ЯЧЕЙКА 5: КЛАСС ДЛЯ ПАРСИНГА ОПИСАНИЯ КОМПАНИИ
class CompanyDescriptionParser:
    """Класс для обработки файла с описанием компании"""

    def __init__(self):
        self.logger = logging.getLogger(f"{__name__}.CompanyDescriptionParser")

    def parse_description(self, file_path: str) -> str:
        """Парсинг описания компании из файла с улучшенной обработкой"""
        try:
            if not os.path.exists(file_path):
                raise FileNotFoundError(f"Файл не найден: {file_path}")

            file_extension = Path(file_path).suffix.lower()
            self.logger.info(f"📖 Парсинг файла: {file_path} (тип: {file_extension})")

            if file_extension == '.docx':
                return self._parse_docx(file_path)
            elif file_extension == '.pdf':
                return self._parse_pdf(file_path)
            elif file_extension == '.txt':
                return self._parse_txt(file_path)
            else:
                raise ValueError(f"❌ Неподдерживаемый формат файла: {file_extension}")

        except Exception as e:
            self.logger.error(f"❌ Ошибка парсинга файла: {str(e)}")
            raise

    def _parse_docx(self, file_path: str) -> str:
        """Парсинг DOCX файла с улучшенной обработкой"""
        try:
            doc = DocxDocument(file_path)
            full_text = []

            for paragraph in doc.paragraphs:
                if paragraph.text.strip():
                    full_text.append(paragraph.text)

            # Добавляем текст из таблиц
            for table in doc.tables:
                for row in table.rows:
                    for cell in row.cells:
                        if cell.text.strip():
                            full_text.append(cell.text)

            text = '\n'.join(full_text)
            self.logger.info(f"✅ DOCX файл обработан: {len(text)} символов")
            return text

        except Exception as e:
            self.logger.error(f"❌ Ошибка парсинга DOCX: {str(e)}")
            raise

    def _parse_pdf(self, file_path: str) -> str:
        """Парсинг PDF файла с улучшенной обработкой"""
        try:
            text = pdf_extract_text(file_path)
            # Очищаем текст от лишних пробелов
            text = re.sub(r'\s+', ' ', text).strip()
            self.logger.info(f"✅ PDF файл обработан: {len(text)} символов")
            return text
        except Exception as e:
            self.logger.error(f"❌ Ошибка парсинга PDF: {str(e)}")
            raise

    def _parse_txt(self, file_path: str) -> str:
        """Парсинг TXT файла с разными кодировками"""
        encodings = ['utf-8', 'windows-1251', 'cp866', 'iso-8859-1']

        for encoding in encodings:
            try:
                with open(file_path, 'r', encoding=encoding) as f:
                    text = f.read()
                self.logger.info(f"✅ TXT файл обработан: {len(text)} символов (кодировка: {encoding})")
                return text
            except UnicodeDecodeError:
                continue

        raise ValueError("❌ Не удалось определить кодировку файла")

    def extract_key_phrases(self, text: str, max_phrases: int = 20) -> List[str]:
        """Извлечение ключевых фраз из текста"""
        # Простая эвристика для извлечения ключевых фраз
        sentences = re.split(r'[.!?]+', text)
        key_phrases = []

        for sentence in sentences:
            sentence = sentence.strip()
            if len(sentence) > 20 and len(sentence) < 200:  # Фильтруем по длине
                words = sentence.split()
                if 3 <= len(words) <= 10:  # Фразы из 3-10 слов
                    key_phrases.append(sentence)

        return key_phrases[:max_phrases]

In [ ]:
# ЯЧЕЙКА 6: ГЕНЕРАТОР КЛЮЧЕВЫХ СЛОВ С MISTRAL AI (ПОЛНОСТЬЮ РАБОЧАЯ ВЕРСИЯ)
import time
import random
import json
import re
from typing import Dict, List, Any
import logging

class MistralKeywordGenerator:
    """Полностью рабочий генератор ключевых слов через Mistral AI"""

    def __init__(self, api_key: str, model: str = "mistral-large-latest"):
        self.client = Mistral(api_key=api_key)
        self.model = model
        self.logger = logging.getLogger(f"{__name__}.MistralKeywordGenerator")

    def generate_keywords(self, company_description: str, max_keywords: int = 300) -> Dict[str, List[str]]:
        """Генерация ключевых слов через Mistral AI с повторными попытками"""
        max_retries = 3
        for attempt in range(max_retries):
            try:
                self.logger.info(f"🔄 Попытка генерации ключевых слов {attempt + 1}/{max_retries}...")

                prompt = self._create_optimized_prompt(company_description, max_keywords)

                # Вызов Mistral без таймаута (убираем параметр timeout)
                response = self.client.chat.complete(
                    model=self.model,
                    messages=[
                        {"role": "user", "content": prompt}
                    ],
                    temperature=0.3
                )

                response_text = response.choices[0].message.content
                self.logger.debug(f"Ответ от Mistral: {response_text}")

                # Извлечение JSON из ответа
                keywords_dict = self._extract_json_from_response(response_text)

                if keywords_dict and self._validate_keywords_dict(keywords_dict):
                    total_keywords = sum(len(v) for v in keywords_dict.values())
                    self.logger.info(f"✅ Сгенерировано {total_keywords} ключевых слов")
                    return keywords_dict
                else:
                    raise ValueError("Не удалось извлечь JSON из ответа Mistral")

            except Exception as e:
                error_msg = str(e)
                self.logger.warning(f"⚠️ Ошибка генерации ключевых слов (попытка {attempt + 1}): {error_msg}")

                # Задержка перед повторной попыткой
                if attempt < max_retries - 1:
                    delay = (2 ** attempt) + random.uniform(1, 3)
                    self.logger.info(f"⏳ Ожидание {delay:.1f} секунд перед повторной попыткой...")
                    time.sleep(delay)

        # Если все попытки не удались, используем резервный метод
        self.logger.error("❌ Все попытки генерации ключевых слов не удались. Используем резервный метод.")
        return self._get_fallback_keywords(company_description)

    def _create_optimized_prompt(self, company_description: str, max_keywords: int) -> str:
        """Создание оптимизированного промпта для Mistral"""
        return f"""
        На основе описания деятельности компании сформируй словарь ключевых слов для поиска релевантных тендеров.

        Описание компании:
        {company_description[:2500]}

        Требования:
        - Сформируй словарь в формате JSON с ключами: "industry", "product", "technical_requirements", "company_type", "materials", "services"
        - В каждом ключе должен быть список до {max_keywords//6} ключевых слов и фраз
        - Слова должны быть на русском языке
        - Слова должны отражать специфику компании и её деятельности
        - Не включай общие слова, только специфичные для отрасли
        - Избегай дубликатов

        Пример ответа:
        {{
            "industry": ["металлургия", "машиностроение", "производство оборудования"],
            "product": ["вал приводной", "шестерня", "подшипник", "тормозная система"],
            "technical_requirements": ["диаметр 120 мм", "точность 0.01 мм", "материал Сталь 45"],
            "company_type": ["производитель", "поставщик", "ООО"],
            "materials": ["сталь", "алюминий", "чугун", "сплавы"],
            "services": ["монтаж", "ремонт", "обслуживание", "наладка"]
        }}

        Верни только JSON без дополнительных пояснений!
        """

    def _extract_json_from_response(self, response_text: str) -> Dict[str, List[str]]:
        """Извлечение JSON из ответа Mistral"""
        try:
            # Убираем возможные лишние символы вокруг JSON
            json_str = response_text.strip().replace('```json', '').replace('```', '')

            # Попробуем найти JSON в ответе
            start = json_str.find('{')
            end = json_str.rfind('}') + 1

            if start != -1 and end != -1:
                json_str = json_str[start:end]
                return json.loads(json_str)
            else:
                # Попробуем извлечь с помощью регулярных выражений
                pattern = r'\{.*\}'
                matches = re.findall(pattern, response_text, re.DOTALL)
                for match in matches:
                    try:
                        return json.loads(match)
                    except json.JSONDecodeError:
                        continue
        except json.JSONDecodeError as e:
            self.logger.error(f"❌ Ошибка парсинга JSON: {str(e)}")
        except Exception as e:
            self.logger.error(f"❌ Неожиданная ошибка при извлечении JSON: {str(e)}")

        return None

    def _validate_keywords_dict(self, keywords_dict: Dict[str, List[str]]) -> bool:
        """Валидация словаря ключевых слов"""
        if not isinstance(keywords_dict, dict):
            return False

        required_keys = ["industry", "product", "technical_requirements", "company_type", "materials", "services"]

        # Проверяем наличие всех требуемых ключей
        for key in required_keys:
            if key not in keywords_dict:
                return False
            if not isinstance(keywords_dict[key], list):
                return False
            if len(keywords_dict[key]) == 0:
                return False

        return True

    def _get_fallback_keywords(self, description: str) -> Dict[str, List[str]]:
        """Резервный метод генерации ключевых слов на основе анализа текста"""
        self.logger.info("🔄 Использование резервного метода генерации ключевых слов")

        # Извлекаем слова из описания
        words = re.findall(r'[а-яёa-zA-Z]{3,}', description.lower())

        # Удаляем стоп-слова
        stop_words = {
            'компания', 'организация', 'предприятие', 'фирма', 'объединение', 'группа', 'холдинг',
            'производство', 'изготовление', 'поставка', 'продажа', 'обслуживание', 'монтаж',
            'ремонт', 'разработка', 'проектирование', 'строительство', 'услуги', 'деятельность',
            'предлагает', 'осуществляет', 'занимается', 'специализируется', 'работает', 'сотрудничает',
            'имеет', 'обладает', 'владеет', 'наш', 'наши', 'ваш', 'ваше', 'его', 'ее', 'их', 'этот',
            'эта', 'это', 'эти', 'тот', 'та', 'то', 'те', 'все', 'всё', 'всех', 'всем', 'всё'
        }

        # Фильтруем слова
        filtered_words = [word for word in words if word not in stop_words and len(word) > 3]

        from collections import Counter
        common_words = [word for word, count in Counter(filtered_words).most_common(50) if count > 1]

        # Классифицируем слова по категориям на основе анализа текста
        industry_keywords = []
        product_keywords = []
        technical_keywords = []
        company_keywords = []
        material_keywords = []
        service_keywords = []

        # Определяем ключевые слова по содержанию
        desc_lower = description.lower()

        if any(word in desc_lower for word in ['металл', 'стал', 'чугун', 'алюмин', 'железо', 'сплав']):
            material_keywords.extend(['сталь', 'алюминий', 'чугун', 'сплавы', 'металл'])

        if any(word in desc_lower for word in ['машиностро', 'оборудование', 'деталь', 'вал', 'шестер', 'подшипник']):
            product_keywords.extend(['вал', 'шестерня', 'подшипник', 'детали', 'оборудование', 'машины'])

        if any(word in desc_lower for word in ['производство', 'промышленность', 'завод', 'фабрика']):
            industry_keywords.extend(['производство', 'промышленность', 'завод', 'фабрика', 'машиностроение'])

        if any(word in desc_lower for word in ['монтаж', 'ремонт', 'обслуживание', 'наладка']):
            service_keywords.extend(['монтаж', 'ремонт', 'обслуживание', 'наладка', 'установка'])

        if any(word in desc_lower for word in ['ооо', 'оао', 'зао', 'ип']):
            company_keywords.extend(['ООО', 'ОАО', 'ЗАО', 'ИП', 'предприятие', 'компания'])

        if any(word in desc_lower for word in ['диаметр', 'точность', 'материал', 'размер']):
            technical_keywords.extend(['диаметр', 'точность', 'материал', 'размер', 'вес', 'габариты'])

        # Добавляем общие слова как резерв
        if len(common_words) > 0:
            # Добавляем в разные категории по принципу частотности
            for i, word in enumerate(common_words[:10]):
                if i % 6 == 0:
                    industry_keywords.append(word)
                elif i % 6 == 1:
                    product_keywords.append(word)
                elif i % 6 == 2:
                    technical_keywords.append(word)
                elif i % 6 == 3:
                    company_keywords.append(word)
                elif i % 6 == 4:
                    material_keywords.append(word)
                else:
                    service_keywords.append(word)

        # Возвращаем словарь
        return {
            "industry": list(set(industry_keywords))[:20],
            "product": list(set(product_keywords))[:20],
            "technical_requirements": list(set(technical_keywords))[:20],
            "company_type": list(set(company_keywords))[:10],
            "materials": list(set(material_keywords))[:20],
            "services": list(set(service_keywords))[:20]
        }

    def print_keywords_summary(self, keywords_dict: Dict[str, List[str]]):
        """Вывод сводки по сгенерированным ключевым словам"""
        print("\n" + "="*80)
        print("🎯 МЕШОК КЛЮЧЕВЫХ СЛОВ ДЛЯ ПОИСКА ТЕНДЕРОВ")
        print("="*80)

        total_keywords = 0
        for category, keywords in keywords_dict.items():
            count = len(keywords)
            total_keywords += count

            # Преобразуем название категории в читаемый формат
            category_names = {
                "industry": "📂 ОТРАСЛИ ПРОМЫШЛЕННОСТИ",
                "product": "📦 ПРОДУКТЫ И УСЛУГИ",
                "technical_requirements": "⚙️ ТЕХНИЧЕСКИЕ ТРЕБОВАНИЯ",
                "company_type": "🏢 ТИП КОМПАНИИ",
                "materials": "🔩 МАТЕРИАЛЫ И СЫРЬЕ",
                "services": "🛠️ УСЛУГИ И РАБОТЫ"
            }

            category_name = category_names.get(category, category.upper())
            print(f"\n{category_name} ({count}):")
            if keywords:
                # Выводим ключевые слова в несколько колонок
                chunk_size = 4
                for i in range(0, len(keywords), chunk_size):
                    chunk = keywords[i:i + chunk_size]
                    print("   " + " | ".join(f"{word:<25}" for word in chunk))

        print(f"\n📊 ВСЕГО КЛЮЧЕВЫХ СЛОВ: {total_keywords}")
        print("="*80)

        # Оценка качества
        if total_keywords < 30:
            print("⚠️  ВНИМАНИЕ: Сгенерировано мало ключевых слов")
        elif total_keywords > 100:
            print("✅ Отличный набор ключевых слов для поиска!")
        else:
            print("✅ Хороший набор ключевых слов")

In [ ]:
# ЯЧЕЙКА 7: КОМПЛЕКСНЫЙ АНАЛИЗАТОР ТЕНДЕРОВ
class TenderAnalyzer:
    """Комплексный анализатор тендеров с использованием spaCy и Mistral AI"""

    def __init__(self, keywords_dict: Dict[str, List[str]], mistral_client=None):
        self.keywords_dict = keywords_dict
        self.mistral_client = mistral_client
        self.logger = logging.getLogger(f"{__name__}.TenderAnalyzer")

        # Загружаем модель spaCy
        try:
            self.nlp = spacy.load("ru_core_news_sm")
            self.logger.info("✅ Модель spaCy загружена успешно")
        except OSError:
            self.logger.warning("⚠️ Модель spaCy не найдена, используем базовый анализ")
            self.nlp = None

        # Создаем объединенный список всех ключевых слов
        self.all_keywords = []
        for keywords in self.keywords_dict.values():
            self.all_keywords.extend(keywords)

        # Создаем паттерны для поиска
        self.keyword_patterns = self._create_keyword_patterns()

        self.logger.info(f"🔍 Загружено {len(self.all_keywords)} ключевых слов для анализа")

    def _create_keyword_patterns(self) -> List[Dict]:
        """Создание паттернов для поиска ключевых слов"""
        patterns = []
        for keyword in self.all_keywords:
            if len(keyword) > 2:  # Игнорируем слишком короткие слова
                patterns.append({
                    "label": "KEYWORD",
                    "pattern": keyword.lower(),
                    "keyword": keyword
                })
        return patterns

    def analyze_tender(self, tender_data: Dict) -> Dict[str, Any]:
        """Комплексный анализ тендера на релевантность"""
        try:
            # Извлекаем текст для анализа
            text = self._extract_tender_text(tender_data)
            if not text or len(text) < 50:
                return self._create_empty_analysis("Мало текста для анализа")

            # Базовый анализ по ключевым словам
            basic_analysis = self._basic_keyword_analysis(text, tender_data)

            # Углубленный анализ с spaCy если доступен
            if self.nlp:
                spacy_analysis = self._spacy_enhanced_analysis(text, tender_data)
                basic_analysis.update(spacy_analysis)

            # Анализ с Mistral AI если доступен
            if self.mistral_client and basic_analysis.get('relevance_score', 0) > 0.1:
                mistral_analysis = self._mistral_deep_analysis(text, tender_data, basic_analysis)
                basic_analysis.update(mistral_analysis)

            return basic_analysis

        except Exception as e:
            self.logger.error(f"❌ Ошибка анализа тендера: {str(e)}")
            return self._create_empty_analysis(f"Ошибка анализа: {str(e)}")

    def _extract_tender_text(self, tender_data: Dict) -> str:
        """Извлечение и объединение текстовой информации из тендера"""
        text_parts = []

        # Основные текстовые поля
        text_fields = [
            'name', 'description', 'purchase_object_info', 'customer_name',
            'lot_name', 'lot_description', 'orderNumber', 'tenderTypeName'
        ]

        for field in text_fields:
            value = tender_data.get(field)
            if value and isinstance(value, str):
                text_parts.append(value)

        # Обработка вложенных структур
        if tender_data.get('lots'):
            for lot in tender_data['lots']:
                text_parts.extend([str(lot.get('name', '')), str(lot.get('description', ''))])

        if tender_data.get('products'):
            for product in tender_data['products']:
                text_parts.extend([str(product.get('name', '')), str(product.get('description', ''))])

        # Объединяем и очищаем текст
        full_text = ' '.join(text_parts)
        full_text = re.sub(r'\s+', ' ', full_text).strip().lower()

        self.logger.debug(f"📝 Извлечен текст длиной {len(full_text)} символов")
        return full_text

    def _basic_keyword_analysis(self, text: str, tender_data: Dict) -> Dict[str, Any]:
        """Базовый анализ по точному совпадению ключевых слов"""
        matched_keywords = []
        text_lower = text.lower()

        # Поиск точных совпадений с приоритетом длинным фразам
        sorted_keywords = sorted(self.all_keywords, key=len, reverse=True)

        for keyword in sorted_keywords:
            keyword_lower = keyword.lower()
            if keyword_lower in text_lower:
                # Находим контекст
                start_pos = text_lower.find(keyword_lower)
                context = self._extract_context(text, start_pos, len(keyword))

                matched_keywords.append({
                    "keyword": keyword,
                    "category": self._find_keyword_category(keyword),
                    "context": context,
                    "match_type": "exact",
                    "score": self._calculate_keyword_score(keyword)
                })

        # Расчет общей оценки релевантности
        total_score = sum(match["score"] for match in matched_keywords)
        max_possible_score = sum(self._calculate_keyword_score(kw) for kw in self.all_keywords[:100])  # Ограничиваем для нормализации
        relevance_score = total_score / max(1, max_possible_score)

        # Учитываем дополнительные факторы
        relevance_score = self._apply_adjustments(relevance_score, tender_data, matched_keywords)

        return {
            "relevance_score": min(relevance_score, 1.0),
            "is_relevant": relevance_score >= 0.15,  # Повышаем порог
            "matched_keywords": matched_keywords,
            "total_matches": len(matched_keywords),
            "unique_matches": len(set(match["keyword"] for match in matched_keywords)),
            "analysis_method": "basic_keyword_matching"
        }

    def _spacy_enhanced_analysis(self, text: str, tender_data: Dict) -> Dict[str, Any]:
        """Улучшенный анализ с использованием spaCy"""
        try:
            doc = self.nlp(text)
            enhanced_matches = []

            # Анализ именованных сущностей
            entities = [ent.text for ent in doc.ents if ent.label_ in ["ORG", "PRODUCT", "LAW"]]

            # Семантический поиск похожих слов
            for keyword in self.all_keywords[:50]:  # Ограничиваем для производительности
                keyword_doc = self.nlp(keyword)
                similarity = doc.similarity(keyword_doc)

                if similarity > 0.7:  # Порог семантической схожести
                    enhanced_matches.append({
                        "keyword": keyword,
                        "similarity": similarity,
                        "match_type": "semantic"
                    })

            return {
                "spacy_entities": entities,
                "semantic_matches": enhanced_matches,
                "analysis_method": "spacy_enhanced"
            }

        except Exception as e:
            self.logger.warning(f"⚠️ Ошибка spaCy анализа: {str(e)}")
            return {}

    def _mistral_deep_analysis(self, text: str, tender_data: Dict, basic_analysis: Dict) -> Dict[str, Any]:
        """Глубокий анализ релевантности с помощью Mistral AI"""
        if not self.mistral_client:
            return {}

        try:
            prompt = self._create_mistral_analysis_prompt(text, tender_data, basic_analysis)

            response = self.mistral_client.chat.complete(
                model="mistral-large-latest",
                messages=[{"role": "user", "content": prompt}],
                temperature=0.2,
                max_tokens=1000
            )

            analysis_text = response.choices[0].message.content
            return self._parse_mistral_analysis(analysis_text)

        except Exception as e:
            self.logger.warning(f"⚠️ Ошибка Mistral анализа: {str(e)}")
            return {}

    def _create_mistral_analysis_prompt(self, text: str, tender_data: Dict, basic_analysis: Dict) -> str:
        """Создание промпта для глубокого анализа"""
        matched_keywords = basic_analysis.get('matched_keywords', [])

        return f"""
        ПРОАНАЛИЗИРУЙ ТЕНДЕР НА РЕЛЕВАНТНОСТЬ ДЛЯ КОМПАНИИ

        ДАННЫЕ ТЕНДЕРА:
        Название: {tender_data.get('name', 'Не указано')}
        Описание: {text[:1000]}

        СОВПАВШИЕ КЛЮЧЕВЫЕ СЛОВА: {[match['keyword'] for match in matched_keywords[:10]]}

        ОЦЕНИ:
        1. Степень соответствия деятельности компании (0-1)
        2. Конкретные аргументы за/против релевантности
        3. Рекомендации по участию

        ОТВЕТ В ФОРМАТЕ JSON:
        {{
            "expert_score": 0.85,
            "relevance_reasons": ["причина1", "причина2"],
            "concerns": ["проблема1", "проблема2"],
            "recommendation": "рекомендация"
        }}
        """

    def _parse_mistral_analysis(self, analysis_text: str) -> Dict[str, Any]:
        """Парсинг анализа от Mistral"""
        try:
            json_match = re.search(r'\{.*\}', analysis_text, re.DOTALL)
            if json_match:
                return json.loads(json_match.group())
        except:
            pass
        return {}

    def _find_keyword_category(self, keyword: str) -> str:
        """Определение категории ключевого слова"""
        for category, keywords in self.keywords_dict.items():
            if keyword in keywords:
                return category
        return "other"

    def _calculate_keyword_score(self, keyword: str) -> float:
        """Расчет веса ключевого слова"""
        base_score = 1.0
        # Увеличиваем вес для длинных и специфичных терминов
        if len(keyword) > 15:
            base_score *= 1.5
        if any(char.isdigit() for char in keyword):  # Технические спецификации
            base_score *= 1.3
        return base_score

    def _extract_context(self, text: str, position: int, keyword_length: int, context_size: int = 100) -> str:
        """Извлечение контекста вокруг ключевого слова"""
        start = max(0, position - context_size)
        end = min(len(text), position + keyword_length + context_size)

        context = text[start:end]
        if start > 0:
            context = "..." + context
        if end < len(text):
            context = context + "..."

        return context

    def _apply_adjustments(self, score: float, tender_data: Dict, matches: List[Dict]) -> float:
        """Применение корректировок к оценке релевантности"""
        # Бонус за технические термины
        technical_matches = [m for m in matches if m['category'] in ['technical_terms', 'products_services']]
        if technical_matches:
            score *= (1 + len(technical_matches) * 0.1)

        # Бонус за крупные тендеры
        if tender_data.get('initial_price'):
            try:
                price = float(tender_data['initial_price'])
                if price > 1000000:
                    score *= 1.2
            except (ValueError, TypeError):
                pass

        return min(score, 1.0)

    def _create_empty_analysis(self, reason: str) -> Dict[str, Any]:
        """Создание пустого анализа"""
        return {
            "relevance_score": 0.0,
            "is_relevant": False,
            "matched_keywords": [],
            "total_matches": 0,
            "unique_matches": 0,
            "analysis_method": "failed",
            "failure_reason": reason
        }

In [ ]:
# ЯЧЕЙКА 8: ФИНАЛЬНЫЙ ПРОЦЕССОР С MISTRAL AI
class MistralFinalProcessor:
    """Финальная обработка релевантных тендеров через Mistral AI"""

    def __init__(self, api_key: str, model: str = "mistral-large-latest"):
        self.client = Mistral(api_key=api_key)
        self.model = model
        self.logger = logging.getLogger(f"{__name__}.MistralFinalProcessor")

    def process_relevant_tender(self, tender_data: Dict, analysis_result: Dict, company_description: str) -> Dict[str, Any]:
        """Финальный экспертный анализ релевантного тендера"""
        try:
            prompt = self._create_expert_analysis_prompt(tender_data, analysis_result, company_description)

            response = self.client.chat.complete(
                model=self.model,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.2,
                max_tokens=2000
            )

            analysis_text = response.choices[0].message.content
            return self._parse_expert_analysis(analysis_text, tender_data, analysis_result)

        except Exception as e:
            self.logger.error(f"❌ Ошибка финальной обработки: {str(e)}")
            return self._create_fallback_analysis(tender_data, analysis_result)

    def _create_expert_analysis_prompt(self, tender_data: Dict, analysis_result: Dict, company_description: str) -> str:
        """Создание промпта для экспертного анализа"""
        tender_info = {
            "Название": tender_data.get('name', 'Не указано'),
            "Заказчик": tender_data.get('customer_name', 'Не указан'),
            "Начальная цена": tender_data.get('initial_price', 'Не указана'),
            "Срок подачи": tender_data.get('submission_deadline', 'Не указан'),
            "Описание": tender_data.get('description', '')[:1500]
        }

        matched_keywords = analysis_result.get('matched_keywords', [])

        return f"""
        ЭКСПЕРТНЫЙ АНАЛИЗ ТЕНДЕРА ДЛЯ КОМПАНИИ

        ОПИСАНИЕ КОМПАНИИ:
        {company_description[:2000]}

        ИНФОРМАЦИЯ О ТЕНДЕРЕ:
        {json.dumps(tender_info, ensure_ascii=False, indent=2)}

        РЕЗУЛЬТАТЫ ПРЕДВАРИТЕЛЬНОГО АНАЛИЗА:
        - Оценка релевантности: {analysis_result.get('relevance_score', 0):.2f}
        - Найдено совпадений: {analysis_result.get('total_matches', 0)}
        - Ключевые совпадения: {[match['keyword'] for match in matched_keywords[:10]]}

        ПРОВЕДИ ДЕТАЛЬНЫЙ АНАЛИЗ И ДАЙ РЕКОМЕНДАЦИИ:

        1. СТЕПЕНЬ СООТВЕТСТВИЯ - насколько тендер соответствует профилю компании
        2. КОНКУРЕНТНЫЕ ПРЕИМУЩЕСТВА - сильные стороны компании для этого тендера
        3. РИСКИ И СЛОЖНОСТИ - возможные проблемы при участии
        4. СТРАТЕГИЯ УЧАСТИЯ - конкретные рекомендации по подготовке
        5. ПРИОРИТЕТНОСТЬ - насколько срочно нужно реагировать

        ФОРМАТ ОТВЕТА (JSON):
        {{
            "expert_score": 0.85,
            "confidence_level": "high",
            "key_advantages": ["преимущество1", "преимущество2"],
            "potential_risks": ["риск1", "риск2"],
            "participation_recommendation": "strongly_recommend",
            "preparation_timeline": "1-2 недели",
            "priority_level": "high",
            "detailed_analysis": "Текст развернутого анализа...",
            "next_steps": ["шаг1", "шаг2", "шаг3"]
        }}

        Будь конкретен и практичен в рекомендациях!
        """

    def _parse_expert_analysis(self, analysis_text: str, tender_data: Dict, analysis_result: Dict) -> Dict[str, Any]:
        """Парсинг экспертного анализа"""
        try:
            json_match = re.search(r'\{.*\}', analysis_text, re.DOTALL)
            if json_match:
                expert_analysis = json.loads(json_match.group())
            else:
                expert_analysis = {"text_analysis": analysis_text}

            return {
                "tender_id": tender_data.get('id'),
                "tender_name": tender_data.get('name'),
                "tender_customer": tender_data.get('customer_name'),
                "initial_price": tender_data.get('initial_price'),
                "initial_analysis": analysis_result,
                "expert_analysis": expert_analysis,
                "processing_time": datetime.now().isoformat()
            }

        except json.JSONDecodeError:
            self.logger.warning("⚠️ Не удалось распарсить JSON экспертного анализа")
            return {
                "tender_id": tender_data.get('id'),
                "tender_name": tender_data.get('name'),
                "initial_analysis": analysis_result,
                "expert_analysis": {"text_analysis": analysis_text},
                "processing_time": datetime.now().isoformat()
            }

    def _create_fallback_analysis(self, tender_data: Dict, analysis_result: Dict) -> Dict[str, Any]:
        """Резервный анализ при ошибке"""
        return {
            "tender_id": tender_data.get('id'),
            "tender_name": tender_data.get('name'),
            "initial_analysis": analysis_result,
            "expert_analysis": {
                "expert_score": analysis_result.get('relevance_score', 0),
                "confidence_level": "medium",
                "key_advantages": ["Автоматический анализ - требуется проверка"],
                "potential_risks": ["Недостаточно данных для полного анализа"],
                "participation_recommendation": "review",
                "priority_level": "medium",
                "detailed_analysis": "Автоматический анализ выполнен успешно, но требуется ручная проверка экспертом",
                "next_steps": ["Изучить документацию тендера", "Оценить ресурсы для участия", "Проверить требования заказчика"]
            },
            "processing_time": datetime.now().isoformat()
        }

In [ ]:
# ЯЧЕЙКА 9: ГЕНЕРАТОР CSV ОТЧЕТОВ
class CSVReportGenerator:
    """Улучшенный генератор CSV отчетов"""

    def __init__(self, output_file: str = "relevant_tenders_report.csv"):
        self.output_file = output_file
        self.logger = logging.getLogger(f"{__name__}.CSVReportGenerator")

    def generate_report(self, processed_tenders: List[Dict]) -> str:
        """Генерация комплексного отчета"""
        try:
            if not processed_tenders:
                self.logger.warning("⚠️ Нет данных для генерации отчета")
                return ""

            with open(self.output_file, 'w', newline='', encoding='utf-8-sig') as csvfile:
                fieldnames = [
                    'tender_id', 'tender_name', 'customer', 'initial_price', 'currency',
                    'deadline', 'relevance_score', 'expert_score', 'confidence_level',
                    'priority_level', 'participation_recommendation', 'key_advantages',
                    'potential_risks', 'preparation_timeline', 'next_steps',
                    'matched_keywords_count', 'processing_date', 'tender_link'
                ]

                writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
                writer.writeheader()

                for tender in processed_tenders:
                    row = self._create_report_row(tender)
                    writer.writerow(row)

            self.logger.info(f"✅ Отчет сохранен: {self.output_file}")
            return self.output_file

        except Exception as e:
            self.logger.error(f"❌ Ошибка генерации отчета: {str(e)}")
            raise

    def _create_report_row(self, tender_data: Dict) -> Dict[str, Any]:
        """Создание строки отчета"""
        expert_analysis = tender_data.get('expert_analysis', {})
        initial_analysis = tender_data.get('initial_analysis', {})

        # Форматируем списки в строки
        key_advantages = self._format_list(expert_analysis.get('key_advantages', []))
        potential_risks = self._format_list(expert_analysis.get('potential_risks', []))
        next_steps = self._format_list(expert_analysis.get('next_steps', []))

        return {
            'tender_id': tender_data.get('tender_id', 'N/A'),
            'tender_name': self._truncate_text(tender_data.get('tender_name', ''), 100),
            'customer': tender_data.get('tender_customer', 'N/A'),
            'initial_price': tender_data.get('initial_price', 'N/A'),
            'currency': 'RUB',
            'deadline': tender_data.get('submission_deadline', 'N/A'),
            'relevance_score': round(initial_analysis.get('relevance_score', 0), 3),
            'expert_score': round(expert_analysis.get('expert_score', 0), 3),
            'confidence_level': expert_analysis.get('confidence_level', 'medium'),
            'priority_level': expert_analysis.get('priority_level', 'medium'),
            'participation_recommendation': expert_analysis.get('participation_recommendation', 'review'),
            'key_advantages': key_advantages,
            'potential_risks': potential_risks,
            'preparation_timeline': expert_analysis.get('preparation_timeline', 'N/A'),
            'next_steps': next_steps,
            'matched_keywords_count': initial_analysis.get('total_matches', 0),
            'processing_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
            'tender_link': f"https://zakupki.gov.ru/epz/order/notice/ea44/view/common-info.html?regNumber={tender_data.get('tender_id', '')}"
        }

    def _format_list(self, items: List) -> str:
        """Форматирование списка в строку"""
        if not items:
            return ""
        return "; ".join(str(item) for item in items if item)

    def _truncate_text(self, text: str, max_length: int) -> str:
        """Обрезка текста до максимальной длины"""
        if len(text) <= max_length:
            return text
        return text[:max_length-3] + "..."

    def generate_summary_report(self, processed_tenders: List[Dict], output_file: str = "tenders_summary.txt") -> str:
        """Генерация текстового суммарного отчета"""
        try:
            with open(output_file, 'w', encoding='utf-8') as f:
                f.write("ОТЧЕТ ПО РЕЛЕВАНТНЫМ ТЕНДЕРАМ\n")
                f.write("=" * 50 + "\n\n")
                f.write(f"Дата генерации: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
                f.write(f"Всего обработано тендеров: {len(processed_tenders)}\n\n")

                high_priority = [t for t in processed_tenders
                               if t.get('expert_analysis', {}).get('priority_level') == 'high']
                medium_priority = [t for t in processed_tenders
                                 if t.get('expert_analysis', {}).get('priority_level') == 'medium']

                f.write(f"Высокий приоритет: {len(high_priority)} тендеров\n")
                f.write(f"Средний приоритет: {len(medium_priority)} тендеров\n\n")

                # Детали по высокоприоритетным тендерам
                if high_priority:
                    f.write("ТЕНДЕРЫ ВЫСОКОГО ПРИОРИТЕТА:\n")
                    f.write("-" * 40 + "\n")
                    for tender in high_priority:
                        expert = tender.get('expert_analysis', {})
                        f.write(f"\n📋 {tender.get('tender_name', 'N/A')}\n")
                        f.write(f"   Заказчик: {tender.get('tender_customer', 'N/A')}\n")
                        f.write(f"   Цена: {tender.get('initial_price', 'N/A')}\n")
                        f.write(f"   Оценка: {expert.get('expert_score', 0):.2f}\n")
                        f.write(f"   Рекомендация: {expert.get('participation_recommendation', 'N/A')}\n")
                        f.write(f"   Срок подготовки: {expert.get('preparation_timeline', 'N/A')}\n")

            self.logger.info(f"✅ Сводный отчет сохранен: {output_file}")
            return output_file

        except Exception as e:
            self.logger.error(f"❌ Ошибка генерации сводного отчета: {str(e)}")
            return ""

In [ ]:
# ЯЧЕЙКА 10: УЛУЧШЕННЫЙ ПАЙПЛАЙН ОБРАБОТКИ
class TimeoutException(Exception):
    pass

@contextmanager
def time_limit(seconds):
    def signal_handler(signum, frame):
        raise TimeoutException(f"Таймаут {seconds} секунд")
    signal.signal(signal.SIGALRM, signal_handler)
    signal.alarm(seconds)
    try:
        yield
    finally:
        signal.alarm(0)

class TenderProcessingPipeline:
    """Улучшенный пайплайн обработки тендеров с таймаутами и обработкой ошибок"""

    def __init__(self, zakupki_client, mistral_api_key: str):
        self.zakupki_client = zakupki_client
        self.mistral_api_key = mistral_api_key
        self.logger = logging.getLogger(f"{__name__}.TenderProcessingPipeline")

        # Инициализация компонентов
        self.file_downloader = FileDownloader()
        self.company_parser = CompanyDescriptionParser()
        self.keyword_generator = MistralKeywordGenerator(mistral_api_key)
        self.final_processor = MistralFinalProcessor(mistral_api_key)
        self.report_generator = CSVReportGenerator()

    def safe_keyword_generation(self, company_description: str, max_retries: int = 2) -> Dict[str, List[str]]:
        """Безопасная генерация ключевых слов с таймаутом"""
        for attempt in range(max_retries):
            try:
                self.logger.info(f"🔄 Попытка генерации ключевых слов {attempt + 1}/{max_retries}...")
                with time_limit(120):  # 2 минуты таймаут
                    keywords_dict = self.keyword_generator.generate_keywords(company_description)
                    return keywords_dict
            except TimeoutException:
                self.logger.warning(f"⏰ Таймаут генерации ключевых слов (попытка {attempt + 1})")
                if attempt == max_retries - 1:
                    return self.keyword_generator._generate_fallback_keywords(company_description)
            except Exception as e:
                self.logger.warning(f"⚠️ Ошибка генерации ключевых слов: {str(e)}")
                if attempt == max_retries - 1:
                    return self.keyword_generator._generate_fallback_keywords(company_description)

    def process(self, company_file_url: str, search_params: Dict = None, max_tenders: int = 50) -> Dict[str, Any]:
        """Основной процесс обработки тендеров"""
        start_time = time.time()
        results = {
            "status": "processing",
            "total_tenders_found": 0,
            "relevant_tenders": 0,
            "processed_tenders": 0,
            "keywords_generated": 0,
            "processing_time": 0,
            "output_file": "",
            "summary_file": "",
            "errors": []
        }

        try:
            # 1. Загрузка и парсинг описания компании
            print("\n📥 ЭТАП 1: ЗАГРУЗКА ОПИСАНИЯ КОМПАНИИ")
            print("-" * 50)

            try:
                with time_limit(60):
                    file_path = self.file_downloader.download_file(company_file_url)

                if not self.file_downloader.validate_file(file_path):
                    raise ValueError("Загруженный файл пуст или поврежден")

                company_description = self.company_parser.parse_description(file_path)
                print(f"✅ Описание компании загружено ({len(company_description)} символов)")

            except TimeoutException:
                error_msg = "Таймаут загрузки файла компании"
                results["errors"].append(error_msg)
                results["status"] = "file_download_timeout"
                return results
            except Exception as e:
                error_msg = f"Ошибка загрузки файла: {str(e)}"
                results["errors"].append(error_msg)
                results["status"] = "file_download_error"
                return results

            # 2. Генерация ключевых слов
            print("\n🤖 ЭТАП 2: ГЕНЕРАЦИЯ КЛЮЧЕВЫХ СЛОВ")
            print("-" * 50)

            keywords_dict = self.safe_keyword_generation(company_description)
            results["keywords_generated"] = sum(len(v) for v in keywords_dict.values())

            # Выводим мешок ключевых слов
            self.keyword_generator.print_keywords_summary(keywords_dict)
            print(f"✅ Сгенерировано ключевых слов: {results['keywords_generated']}")

            # 3. Поиск тендеров
            print("\n🔍 ЭТАП 3: ПОИСК ТЕНДЕРОВ")
            print("-" * 50)

            # Формируем поисковый запрос из ключевых слов
            search_query = self._create_search_query(keywords_dict)
            search_params = search_params or {}
            search_params.update({
                "query": search_query,
                "per_page": min(max_tenders, 50)  # Ограничиваем для тестов
            })

            try:
                with time_limit(120):
                    tenders = self.zakupki_client.search_tenders(search_params)

                results["total_tenders_found"] = len(tenders)
                print(f"✅ Найдено тендеров: {len(tenders)}")

                if not tenders:
                    results["status"] = "no_tenders_found"
                    return results

            except TimeoutException:
                error_msg = "Таймаут поиска тендеров"
                results["errors"].append(error_msg)
                results["status"] = "search_timeout"
                return results
            except Exception as e:
                error_msg = f"Ошибка поиска тендеров: {str(e)}"
                results["errors"].append(error_msg)
                results["status"] = "search_error"
                return results

            # 4. Анализ тендеров
            print("\n🔎 ЭТАП 4: АНАЛИЗ ТЕНДЕРОВ НА РЕЛЕВАНТНОСТЬ")
            print("-" * 50)

            analyzer = TenderAnalyzer(keywords_dict)
            relevant_tenders = []

            for i, tender in enumerate(tenders):
                try:
                    if i % 10 == 0:
                        print(f"   📊 Анализировано {i}/{len(tenders)} тендеров...")

                    analysis = analyzer.analyze_tender(tender)
                    if analysis.get('is_relevant', False):
                        relevant_tenders.append((tender, analysis))

                except Exception as e:
                    error_msg = f"Ошибка анализа тендера {i}: {str(e)}"
                    results["errors"].append(error_msg)
                    continue

            results["relevant_tenders"] = len(relevant_tenders)
            print(f"✅ Найдено релевантных тендеров: {len(relevant_tenders)}")

            # 5. Углубленный анализ релевантных тендеров
            print("\n🧠 ЭТАП 5: ЭКСПЕРТНЫЙ АНАЛИЗ")
            print("-" * 50)

            final_processed = []
            max_expert_analysis = min(10, len(relevant_tenders))  # Ограничиваем для производительности

            for i, (tender, analysis) in enumerate(relevant_tenders[:max_expert_analysis]):
                try:
                    print(f"   🔍 Экспертный анализ {i+1}/{max_expert_analysis}...")

                    with time_limit(180):
                        expert_analysis = self.final_processor.process_relevant_tender(
                            tender, analysis, company_description
                        )

                    final_processed.append(expert_analysis)
                    self._print_tender_analysis_summary(expert_analysis)

                except TimeoutException:
                    print(f"⏰ Таймаут экспертного анализа тендера {i+1}")
                    continue
                except Exception as e:
                    error_msg = f"Ошибка экспертного анализа: {str(e)}"
                    results["errors"].append(error_msg)
                    continue

            results["processed_tenders"] = len(final_processed)

            # 6. Генерация отчетов
            print("\n📊 ЭТАП 6: ГЕНЕРАЦИЯ ОТЧЕТОВ")
            print("-" * 50)

            if final_processed:
                try:
                    output_file = self.report_generator.generate_report(final_processed)
                    results["output_file"] = output_file

                    summary_file = self.report_generator.generate_summary_report(final_processed)
                    results["summary_file"] = summary_file

                    print(f"✅ Основной отчет: {output_file}")
                    print(f"✅ Сводный отчет: {summary_file}")
                except Exception as e:
                    error_msg = f"Ошибка генерации отчета: {str(e)}"
                    results["errors"].append(error_msg)
            else:
                print("ℹ️ Нет данных для генерации отчетов")

            # 7. Финальная статистика
            results["processing_time"] = time.time() - start_time
            results["status"] = "completed"

            self._print_final_results(results)
            return results

        except KeyboardInterrupt:
            print("\n🛑 Процесс прерван пользователем")
            results["status"] = "interrupted"
            results["processing_time"] = time.time() - start_time
            return results
        except Exception as e:
            print(f"\n💥 Критическая ошибка: {str(e)}")
            results["status"] = "critical_error"
            results["processing_time"] = time.time() - start_time
            return results

    def _create_search_query(self, keywords_dict: Dict[str, List[str]]) -> str:
        """Создание поискового запроса из ключевых слов"""
        # Берем наиболее релевантные ключевые слова
        priority_categories = ["products_services", "equipment", "technical_terms"]
        search_keywords = []

        for category in priority_categories:
            if category in keywords_dict:
                search_keywords.extend(keywords_dict[category][:5])  # Берем по 5 из каждой категории

        # Ограничиваем общее количество и убираем дубликаты
        search_keywords = list(set(search_keywords))[:10]

        # Формируем поисковый запрос
        query = " OR ".join(f'"{kw}"' for kw in search_keywords if len(kw) > 3)
        return query

    def _print_tender_analysis_summary(self, tender_analysis: Dict):
        """Вывод краткой информации об анализе тендера"""
        expert = tender_analysis.get('expert_analysis', {})
        initial = tender_analysis.get('initial_analysis', {})

        tender_name = tender_analysis.get('tender_name', 'Без названия')
        expert_score = expert.get('expert_score', 0)
        priority = expert.get('priority_level', 'medium')
        recommendation = expert.get('participation_recommendation', 'review')

        print(f"   ✅ {tender_name[:60]}...")
        print(f"      Оценка: {initial.get('relevance_score', 0):.2f} → Эксперт: {expert_score:.2f}")
        print(f"      Приоритет: {priority} | Рекомендация: {recommendation}")

    def _print_final_results(self, results: Dict):
        """Вывод финальных результатов"""
        print("\n" + "="*80)
        print("🎉 ФИНАЛЬНЫЕ РЕЗУЛЬТАТЫ ОБРАБОТКИ")
        print("="*80)

        status_icons = {
            "completed": "✅",
            "interrupted": "⚠️",
            "critical_error": "❌",
            "no_tenders_found": "🔍",
            "file_download_error": "📥",
            "search_error": "🔍"
        }

        icon = status_icons.get(results["status"], "❓")

        print(f"{icon} Статус: {results['status']}")
        print(f"📈 Найдено тендеров: {results['total_tenders_found']}")
        print(f"✅ Релевантных: {results['relevant_tenders']}")
        print(f"🔄 Обработано: {results['processed_tenders']}")
        print(f"🔑 Ключевых слов: {results['keywords_generated']}")
        print(f"⏱️ Время обработки: {results['processing_time']:.2f} сек.")

        if results["output_file"]:
            print(f"📊 Отчеты: {results['output_file']}, {results['summary_file']}")

        if results["errors"]:
            print(f"\n⚠️ Ошибок в процессе: {len(results['errors'])}")
            for error in results["errors"][:5]:  # Показываем только первые 5 ошибок
                print(f"   - {error}")

In [ ]:
# ЯЧЕЙКА 11: АУТЕНТИФИКАЦИЯ И ПРОВЕРКА ПОДКЛЮЧЕНИЯ
print("🔐 АУТЕНТИФИКАЦИЯ В СИСТЕМЕ ZAKUPKI360")
print("=" * 50)

# Ввод данных пользователя
zakupki_login = input("Введите логин Zakupki360: ").strip()
zakupki_password = input("Введите пароль Zakupki360: ").strip()
mistral_api_key = input("Введите API ключ Mistral AI: ").strip()

# Проверка аутентификации
try:
    print("\n🔄 Проверка подключения к Zakupki360 API...")
    zakupki_client = Zakupki360APIClient(login=zakupki_login, password=zakupki_password)

    # Тестируем подключение
    if zakupki_client.test_connection():
        print("✅ Подключение к Zakupki360 API успешно!")

        # Показываем статистику использования
        stats = zakupki_client.get_usage_stats()
        print(f"📊 Лимиты API: {stats['remaining']['search']} поисковых запросов осталось")
    else:
        print("❌ Не удалось подключиться к Zakupki360 API")
        print("💡 Проверьте логин, пароль и подключение к интернету")

except Exception as e:
    print(f"❌ Ошибка аутентификации: {str(e)}")
    print("\n🔧 Рекомендации по устранению проблем:")
    print("1. Проверьте правильность логина и пароля")
    print("2. Убедитесь что аккаунт имеет доступ к API")
    print("3. Проверьте подключение к интернету")
    print("4. Попробуйте использовать тестовый режим если доступен")

🔐 АУТЕНТИФИКАЦИЯ В СИСТЕМЕ ZAKUPKI360
Введите логин Zakupki360: <ZAKUPKI360_LOGIN — брать из .env>
Введите пароль Zakupki360: <ZAKUPKI360_PASSWORD — сменён, брать из .env>
Введите API ключ Mistral AI: <MISTRAL_API_KEY — отозван, брать из .env>

🔄 Проверка подключения к Zakupki360 API...
✅ Подключение к Zakupki360 API успешно!
📊 Лимиты API: 99 поисковых запросов осталось


*Введите логин для Zakupki360: <ZAKUPKI360_LOGIN — брать из .env>

*Введите пароль для Zakupki360: <ZAKUPKI360_PASSWORD — сменён, брать из .env>

*Введите API ключ для Mistral: <MISTRAL_API_KEY — отозван, брать из .env>

https://docs.google.com/document/d/1RggP8wiq6zIW0Wwx9qbHUCJOny7z13NmxSRmSdvwBs8/edit?usp=sharing

In [ ]:
# ЯЧЕЙКА 12: ВВОД ДАННЫХ ДЛЯ ПОИСКА
print("\n🎯 НАСТРОЙКА ПАРАМЕТРОВ ПОИСКА")
print("=" * 50)

# Ссылка на файл с описанием компании
company_file_url = input("Введите ссылку на файл с описанием компании (DOCX/PDF/TXT/Google Docs): ").strip()

# Параметры поиска
print("\n📋 ДОПОЛНИТЕЛЬНЫЕ ПАРАМЕТРЫ ПОИСКА:")
search_query = input("Дополнительные ключевые слова для поиска (Enter для пропуска): ").strip()
max_tenders = input("Максимальное количество тендеров для анализа (по умолчанию 30): ").strip()
max_tenders = int(max_tenders) if max_tenders.isdigit() else 30

# Формируем параметры поиска
search_params = {
    "per_page": min(max_tenders, 50),
    "query": search_query if search_query else None
}

print(f"\n🚀 ПАРАМЕТРЫ ПОИСКА:")
print(f"   - Максимальное количество тендеров: {max_tenders}")
print(f"   - Дополнительные ключевые слова: {search_query if search_query else 'не указаны'}")
print(f"   - Источник описания компании: {company_file_url}")


🎯 НАСТРОЙКА ПАРАМЕТРОВ ПОИСКА
Введите ссылку на файл с описанием компании (DOCX/PDF/TXT/Google Docs): https://docs.google.com/document/d/1RggP8wiq6zIW0Wwx9qbHUCJOny7z13NmxSRmSdvwBs8/edit?usp=sharing

📋 ДОПОЛНИТЕЛЬНЫЕ ПАРАМЕТРЫ ПОИСКА:
Дополнительные ключевые слова для поиска (Enter для пропуска): вал вал приводной шестерня
Максимальное количество тендеров для анализа (по умолчанию 30): 30

🚀 ПАРАМЕТРЫ ПОИСКА:
   - Максимальное количество тендеров: 30
   - Дополнительные ключевые слова: вал вал приводной шестерня
   - Источник описания компании: https://docs.google.com/document/d/1RggP8wiq6zIW0Wwx9qbHUCJOny7z13NmxSRmSdvwBs8/edit?usp=sharing


In [ ]:
# ЯЧЕЙКА 13: ЗАПУСК ОСНОВНОГО ПРОЦЕССА
print("\n🚀 ЗАПУСК ПРОЦЕССА АНАЛИЗА ТЕНДЕРОВ")
print("=" * 50)

try:
    # Проверяем что клиент инициализирован
    if 'zakupki_client' not in locals():
        print("❌ Клиент Zakupki360 не инициализирован")
        print("💡 Запустите сначала ячейку аутентификации")
    else:
        # Создаем и запускаем пайплайн
        pipeline = TenderProcessingPipeline(zakupki_client, mistral_api_key)

        print("🎯 Начинаем обработку...")
        print("💡 Это может занять несколько минут")

        # Запускаем процесс
        results = pipeline.process(
            company_file_url=company_file_url,
            search_params=search_params,
            max_tenders=max_tenders
        )

        # Выводим итоговый статус
        if results["status"] == "completed":
            print(f"\n✅ Процесс успешно завершен за {results['processing_time']:.2f} секунд")
            if results["output_file"]:
                print(f"📁 Отчеты сохранены в файлах:")
                print(f"   - Детальный отчет: {results['output_file']}")
                print(f"   - Сводный отчет: {results['summary_file']}")
        else:
            print(f"\n⚠️ Процесс завершен со статусом: {results['status']}")

except Exception as e:
    print(f"\n❌ Ошибка при запуске процесса: {str(e)}")
    print("💡 Проверьте введенные данные и попробуйте снова")


🚀 ЗАПУСК ПРОЦЕССА АНАЛИЗА ТЕНДЕРОВ
🎯 Начинаем обработку...
💡 Это может занять несколько минут

📥 ЭТАП 1: ЗАГРУЗКА ОПИСАНИЯ КОМПАНИИ
--------------------------------------------------
✅ Описание компании загружено (1508 символов)

🤖 ЭТАП 2: ГЕНЕРАЦИЯ КЛЮЧЕВЫХ СЛОВ
--------------------------------------------------

🎯 МЕШОК КЛЮЧЕВЫХ СЛОВ ДЛЯ ПОИСКА ТЕНДЕРОВ

📂 ОТРАСЛИ ПРОМЫШЛЕННОСТИ (43):
   тяжёлое машиностроение    | горнодобывающая промышленность | нефтегазовый комплекс     | химическая промышленность
   энергетическое машиностроение | металлургическая промышленность | промышленное оборудование | перерабатывающая промышленность
   целлюлозно-бумажная промышленность | производство минеральных удобрений | металлообработка          | судостроение             
   авиационная промышленность | космическая промышленность | производство строительных материалов | пищевая промышленность (промышленное оборудование)
   фармацевтическая промышленность (оборудование) | производство стекольной прод

/tmp/ipython-input-1329984066.py:150: UserWarning: [W007] The model you're using has no word vectors loaded, so the result of the Doc.similarity method will be based on the tagger, parser and NER, which may not give useful similarity judgements. This may happen if you're using one of the small models, e.g. `en_core_web_sm`, which don't ship with word vectors and only use context-sensitive tensors. You can always add your own word vectors, or use one of the larger models instead if available.
  similarity = doc.similarity(keyword_doc)


   📊 Анализировано 10/1000 тендеров...
   📊 Анализировано 20/1000 тендеров...
   📊 Анализировано 30/1000 тендеров...
   📊 Анализировано 40/1000 тендеров...
   📊 Анализировано 50/1000 тендеров...
   📊 Анализировано 60/1000 тендеров...
   📊 Анализировано 70/1000 тендеров...
   📊 Анализировано 80/1000 тендеров...
   📊 Анализировано 90/1000 тендеров...
   📊 Анализировано 100/1000 тендеров...
   📊 Анализировано 110/1000 тендеров...
   📊 Анализировано 120/1000 тендеров...
   📊 Анализировано 130/1000 тендеров...
   📊 Анализировано 140/1000 тендеров...
   📊 Анализировано 150/1000 тендеров...
   📊 Анализировано 160/1000 тендеров...
   📊 Анализировано 170/1000 тендеров...
   📊 Анализировано 180/1000 тендеров...
   📊 Анализировано 190/1000 тендеров...
   📊 Анализировано 200/1000 тендеров...
   📊 Анализировано 210/1000 тендеров...
   📊 Анализировано 220/1000 тендеров...
   📊 Анализировано 230/1000 тендеров...
   📊 Анализировано 240/1000 тендеров...
   📊 Анализировано 250/1000 тендеров...
   📊 Анал

In [ ]:
# ЯЧЕЙКА 14: ДОПОЛНИТЕЛЬНЫЕ ФУНКЦИИ И СТАТИСТИКА
def show_detailed_statistics():
    """Показать детальную статистику"""
    if 'zakupki_client' in locals() and 'results' in locals():
        print("\n📊 ДЕТАЛЬНАЯ СТАТИСТИКА")
        print("=" * 50)

        # Статистика API
        api_stats = zakupki_client.get_usage_stats()
        print("🎯 ИСПОЛЬЗОВАНИЕ API:")
        for endpoint in api_stats['remaining']:
            used = api_stats['used'][endpoint]
            remaining = api_stats['remaining'][endpoint]
            print(f"   {endpoint:15} | Использовано: {used:2d} | Осталось: {remaining:3d}")

        # Статистика обработки
        print(f"\n📈 РЕЗУЛЬТАТЫ ОБРАБОТКИ:")
        print(f"   Статус: {results.get('status', 'unknown')}")
        print(f"   Найдено тендеров: {results.get('total_tenders_found', 0)}")
        print(f"   Релевантных: {results.get('relevant_tenders', 0)}")
        print(f"   Обработано: {results.get('processed_tenders', 0)}")
        print(f"   Ключевых слов: {results.get('keywords_generated', 0)}")
        print(f"   Время: {results.get('processing_time', 0):.2f} сек.")

        if results.get('errors'):
            print(f"\n⚠️ ОШИБКИ ({len(results['errors'])}):")
            for error in results['errors'][:3]:
                print(f"   - {error}")

def download_reports():
    """Скачать сгенерированные отчеты"""
    if 'results' in locals():
        try:
            from google.colab import files

            if results.get('output_file') and os.path.exists(results['output_file']):
                files.download(results['output_file'])
                print(f"✅ Отчет {results['output_file']} скачан!")
            else:
                print("❌ Файл отчета не найден")

            if results.get('summary_file') and os.path.exists(results['summary_file']):
                files.download(results['summary_file'])
                print(f"✅ Сводный отчет {results['summary_file']} скачан!")

        except ImportError:
            print("ℹ️ Функция скачивания доступна только в Google Colab")
        except Exception as e:
            print(f"❌ Ошибка скачивания: {str(e)}")
    else:
        print("❌ Нет результатов для скачивания")

def cleanup_files():
    """Очистка временных файлов"""
    files_to_keep = ['tender_processing.log']

    if 'results' in locals():
        if results.get('output_file'):
            files_to_keep.append(results['output_file'])
        if results.get('summary_file'):
            files_to_keep.append(results['summary_file'])

    deleted_count = 0
    for file in os.listdir('.'):
        if (file.startswith('company_description_') or
            file.startswith('downloaded_file_')) and file not in files_to_keep:
            try:
                os.remove(file)
                deleted_count += 1
            except:
                pass

    print(f"✅ Удалено {deleted_count} временных файлов")

# Пример использования дополнительных функций
print("\n🔧 ДОПОЛНИТЕЛЬНЫЕ ФУНКЦИИ")
print("=" * 50)

show_detailed_statistics()

print("\n💾 СКАЧИВАНИЕ ОТЧЕТОВ:")
download_reports()

print("\n🧹 ОЧИСТКА ВРЕМЕННЫХ ФАЙЛОВ:")
cleanup_files()


🔧 ДОПОЛНИТЕЛЬНЫЕ ФУНКЦИИ

💾 СКАЧИВАНИЕ ОТЧЕТОВ:
❌ Нет результатов для скачивания

🧹 ОЧИСТКА ВРЕМЕННЫХ ФАЙЛОВ:
✅ Удалено 1 временных файлов
